# 🌪️ CATALYST RISK: Mathematical Dynamics & Deep Dive
## Advanced Stochastic Catastrophe Modelling

This interactive notebook demonstrates the core mathematical and statistical dynamics powering the CATALYST RISK engine. 
It features highly complex, real-world visualizations used in tier-1 quantitative risk modeling, spanning 3D vulnerability surfaces, financial reconciliation waterfalls, and probabilistic spatial density mapping.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Set dark theme to match CATALYST UI
pio.templates.default = 'plotly_dark'

### 1. Stochastic Hazard Dynamics (Probability Density Functions)

In [ ]:
def plot_hazard_distributions():
    np.random.seed(42)
    sims = 100000
    
    scenarios = {
        'Moderate (Baseline)': {'base': 100, 'sigma': 0.32},
        'High (+30%)': {'base': 130, 'sigma': 0.38},
        'Extreme Tail (+65%)': {'base': 165, 'sigma': 0.45}
    }
    
    fig = go.Figure()
    colors = ['#0ea5e9', '#f59e0b', '#ef4444']
    
    for (name, params), color in zip(scenarios.items(), colors):
        mu = np.log(params['base']) - (params['sigma']**2 / 2)
        intensities = np.random.lognormal(mean=mu, sigma=params['sigma'], size=sims)
        
        hist, bin_edges = np.histogram(intensities, bins=200, range=(0, 400), density=True)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        fig.add_trace(go.Scatter(
            x=bin_centers, y=hist, mode='lines', fill='tozeroy', 
            name=name, line=dict(color=color, width=2), opacity=0.6
        ))

    fig.update_layout(
        title='Stochastic Hazard Intensity Distributions (Probability Density)',
        xaxis_title='Hazard Intensity (e.g., mph, PGA)',
        yaxis_title='Probability Density',
        legend=dict(x=0.7, y=0.9), height=500
    )
    return fig

plot_hazard_distributions().show()

### 2. Vulnerability Engineering (Bounded Logistic Response)

In [ ]:
def calculate_logistic_damage(intensity, mid, k, cap):
    dr = cap / (1.0 + np.exp(-k * (intensity - mid)))
    return np.clip(dr, 0, cap)

def plot_vulnerability_curves():
    intensities = np.linspace(0, 300, 500)
    vuln_params = {
        'Wood Frame': {'mid': 90.0, 'k': 0.045, 'cap': 0.97, 'color': '#d97706'},
        'Masonry': {'mid': 105.0, 'k': 0.038, 'cap': 0.92, 'color': '#ef4444'},
        'Reinforced Concrete': {'mid': 140.0, 'k': 0.032, 'cap': 0.85, 'color': '#3b82f6'},
        'Engineered Steel': {'mid': 155.0, 'k': 0.030, 'cap': 0.70, 'color': '#10b981'}
    }
    
    fig = go.Figure()
    for material, params in vuln_params.items():
        dr = calculate_logistic_damage(intensities, params['mid'], params['k'], params['cap'])
        fig.add_trace(go.Scatter(x=intensities, y=dr, mode='lines', name=material, line=dict(width=3, color=params['color'])))
        
    fig.add_vline(x=90, line_dash='dot', line_color='rgba(255,255,255,0.2)')
    fig.add_vline(x=155, line_dash='dot', line_color='rgba(255,255,255,0.2)')

    fig.update_layout(
        title='Structural Vulnerability Response Functions',
        xaxis_title='Hazard Intensity', yaxis_title='Mean Damage Ratio (MDR)',
        yaxis=dict(tickformat='.0%'), height=500
    )
    return fig

plot_vulnerability_curves().show()

### 3. Bivariate 3D Vulnerability Surface (Intensity vs. Construction Quality)
Real-world vulnerability is highly multi-dimensional. Here we plot the Damage Ratio as a function of both the Hazard Intensity and a Construction Quality Multiplier.

In [ ]:
def plot_3d_vulnerability():
    intensity = np.linspace(0, 250, 100)
    quality = np.linspace(0.5, 1.5, 100)
    I, Q = np.meshgrid(intensity, quality)
    
    # Complex multi-variate formula: Quality scales the inflection point and cap
    mid = 120 * Q
    cap = np.clip(0.95 / Q, 0, 1)
    k = 0.035
    
    DR = cap / (1.0 + np.exp(-k * (I - mid)))
    
    fig = go.Figure(data=[go.Surface(z=DR, x=I, y=Q, colorscale='Inferno', opacity=0.9)])
    fig.update_layout(
        title='3D Bivariate Vulnerability Surface',
        scene=dict(
            xaxis_title='Hazard Intensity',
            yaxis_title='Construction Quality (Multiplier)',
            zaxis_title='Damage Ratio',
            camera=dict(eye=dict(x=1.8, y=-1.8, z=0.8))
        ),
        height=700
    )
    return fig

plot_3d_vulnerability().show()

### 4. Portfolio Financial Reconciliation (Waterfall Chart)
Visualizing how Ground-Up Loss (GUL) transforms into Net Loss after accounting for policy retentions (deductibles) and exhaustion caps (limits).

In [ ]:
def plot_financial_waterfall():
    fig = go.Figure(go.Waterfall(
        name='Financial Leakage', orientation='v',
        measure=['relative', 'relative', 'relative', 'total'],
        x=['Ground-Up Loss (GUL)', 'Deductible (Retained)', 'Policy Limits (Capped)', 'Net Insured Loss'],
        textposition='outside',
        text=['$850M', '-$120M', '-$210M', '$520M'],
        y=[850, -120, -210, 520],
        connector={'line':{'color':'rgb(63, 63, 63)'}},
        decreasing={'marker':{'color':'#10b981'}},
        increasing={'marker':{'color':'#ef4444'}},
        totals={'marker':{'color':'#0ea5e9'}}
    ))
    fig.update_layout(title='Event Reinsurance Financial Reconciliation (Gross to Net)', height=500, showlegend=False)
    return fig

plot_financial_waterfall().show()

### 5. Probabilistic Loss Variance (Violin Plots)
Visualizing the complete probability density of losses across different construction materials using Violin plots (combining KDE and Box plots).

In [ ]:
def plot_probabilistic_violins():
    np.random.seed(7)
    n_sims = 2000
    
    data = []
    for mat, base_mu in zip(['Concrete', 'Steel', 'Masonry', 'Wood'], [9, 9.5, 11, 12]):
        # Generating skewed synthetic losses per material
        losses = np.random.lognormal(mean=base_mu, sigma=1.2, size=n_sims)
        df = pd.DataFrame({'Loss': losses, 'Material': mat})
        data.append(df)
        
    df_all = pd.concat(data)
    
    fig = px.violin(df_all, y='Loss', color='Material', box=True, points='outliers', 
                    title='Loss Distribution Variance by Structural Material',
                    color_discrete_sequence=['#3b82f6', '#10b981', '#ef4444', '#d97706'])
    
    fig.update_layout(yaxis_type='log', yaxis_title='Simulated Loss (Log Scale, USD)', height=600)
    return fig

plot_probabilistic_violins().show()

### 6. Geospatial Risk Concentration (Heatmap Density)
Mapping the aggregate expected loss concentrations spatially to identify 'hotspots' in the portfolio.

In [ ]:
def plot_spatial_density():
    # Generate synthetic clustered exposure in California
    np.random.seed(42)
    lat = np.random.normal(34.05, 0.5, 3000)
    lon = np.random.normal(-118.24, 0.5, 3000)
    losses = np.random.lognormal(mean=10, sigma=1, size=3000)
    
    df = pd.DataFrame({'Lat': lat, 'Lon': lon, 'Loss': losses})
    
    fig = px.density_mapbox(df, lat='Lat', lon='Lon', z='Loss', radius=15,
                            center=dict(lat=34.05, lon=-118.24), zoom=7,
                            mapbox_style='carto-darkmatter', 
                            title='Geospatial Expected Loss Density (California Portfolio)')
    fig.update_layout(height=600)
    return fig

plot_spatial_density().show()

### 7. The OEP Curve (Occurrence Exceedance Probability)

In [ ]:
def plot_complex_ep_curve():
    np.random.seed(99)
    sims = 10000
    occurrences = np.random.rand(sims) < 0.15
    losses = np.random.lognormal(mean=14, sigma=1.2, size=sims) * occurrences
    sorted_losses = np.sort(losses)[::-1]
    
    probabilities = np.arange(1, sims + 1) / sims
    return_periods = 1.0 / probabilities
    
    mask = sorted_losses > 0
    rp_filtered = return_periods[mask]
    loss_filtered = sorted_losses[mask]
    
    aal = np.mean(losses)
    pml100 = sorted_losses[int(np.floor(sims / 100)) - 1]
    pml250 = sorted_losses[int(np.floor(sims / 250)) - 1]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rp_filtered, y=loss_filtered, mode='lines', name='OEP Curve', line=dict(color='#ef4444', width=3)))
    fig.add_hline(y=aal, line_dash='dash', line_color='#10b981', annotation_text=f'AAL: ${aal:,.0f}', annotation_position='bottom right')
    fig.add_vline(x=100, line_dash='dash', line_color='#f59e0b', annotation_text=f'100-Year PML<br>${pml100:,.0f}')
    fig.add_vline(x=250, line_dash='dash', line_color='#ef4444', annotation_text=f'250-Year PML<br>${pml250:,.0f}')

    fig.update_layout(title='Occurrence Exceedance Probability (OEP) - Tail Risk Analysis', xaxis_title='Return Period (Years)', yaxis_title='Aggregate Portfolio Loss ($)', xaxis_type='log', height=600)
    return fig

plot_complex_ep_curve().show()

### 8. Model Sensitivity (Tornado Analysis)
Analyzing how parametric shocks (e.g., climate change increasing hazard severity, or inflation shifting deductibles) impact the 250-Year PML.

In [ ]:
def plot_tornado_chart():
    scenarios = [
        'Climate Shift (+15% Hazard)', 
        'Inflation (+10% TIV)', 
        'Deductible Decrease (-20%)', 
        'Vulnerability Shift (+5%)', 
        'Strict Limits (-10% Policy Cap)'
    ]
    # Percentage change in 250-Year PML
    impacts = [18.5, 10.0, 4.2, 8.7, -12.4]
    
    fig = go.Figure()
    
    # Color positive impacts red (bad for PML), negative impacts green (good for PML)
    colors = ['#ef4444' if val > 0 else '#10b981' for val in impacts]
    
    fig.add_trace(go.Bar(
        y=scenarios,
        x=impacts,
        orientation='h',
        marker_color=colors,
        text=[f'+{v}%' if v>0 else f'{v}%' for v in impacts],
        textposition='outside'
    ))
    
    fig.update_layout(
        title='Sensitivity Tornado Chart (Impact on 250-Year PML)',
        xaxis_title='Change in 250-Year PML (%)',
        yaxis_title='Parametric Shock Scenario',
        yaxis=dict(autorange='reversed'),
        height=500
    )
    return fig

plot_tornado_chart().show()